# Explanation — Assignment 1 (AI for Robotics)

Study notes with rendered math. Run/open in Jupyter (or VS Code with the venv kernel) — markdown cells render LaTeX automatically.

## Task 1.2 — Rotation (detailed)

### 1. The concept
A **rotation** turns a body about the **origin** by an angle $\theta$ (counterclockwise for $\theta > 0$), **without changing its size or shape**. Every point keeps its distance from the origin; only its direction changes.

### 2. The rotation matrix
A counterclockwise rotation by $\theta$ is the linear map

$$
R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix},
\qquad
\begin{pmatrix} x' \\ y' \end{pmatrix} = R(\theta)\begin{pmatrix} x \\ y \end{pmatrix}.
$$

**Where do the entries come from?** A matrix is fully defined by what it does to the basis vectors:

$$
\begin{pmatrix} 1 \\ 0 \end{pmatrix} \;\longmapsto\; \begin{pmatrix} \cos\theta \\ \sin\theta \end{pmatrix}\;\;(\text{1st column}),
\qquad
\begin{pmatrix} 0 \\ 1 \end{pmatrix} \;\longmapsto\; \begin{pmatrix} -\sin\theta \\ \cos\theta \end{pmatrix}\;\;(\text{2nd column}).
$$

Quick check at $\theta = 90^\circ$: $(1,0)\mapsto(0,1)$ and $(0,1)\mapsto(-1,0)$ — a quarter-turn CCW. ✅

### 3. Why the code uses `polygon @ R.T`
The formula $\mathbf{p}' = R\,\mathbf{p}$ assumes $\mathbf{p}$ is a **column** vector. But we store the polygon as an $(N,2)$ array — **one point per row**. Transposing the identity gives the *row* form:

$$
(R\mathbf{p})^\top = \mathbf{p}^\top R^\top
\qquad\Longrightarrow\qquad
\begin{pmatrix} x' & y' \end{pmatrix} = \begin{pmatrix} x & y \end{pmatrix} R^\top .
$$

So **for row vectors we right-multiply by $R^\top$** — that is the `.T` in `polygon @ R.T`. Because the polygon stacks $N$ rows, this rotates all $N$ vertices at once and returns an $(N,2)$ array.

Explicitly,

$$
R^\top = \begin{pmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{pmatrix},
\qquad
\begin{pmatrix} x & y \end{pmatrix} R^\top
= \begin{pmatrix}\, x\cos\theta - y\sin\theta \;\;&\;\; x\sin\theta + y\cos\theta \,\end{pmatrix}.
$$

> ⚠️ **Classic trap:** writing `polygon @ R` (no `.T`) silently applies the **clockwise** rotation $R(-\theta)$. The picture still looks "rotated," just the wrong way.

### 4. Worked example — rotate $(2,0)$ by $90^\circ$
At $\theta = 90^\circ$: $\cos\theta = 0,\ \sin\theta = 1$, so $R^\top = \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix}$.

$$
\begin{pmatrix} 2 & 0 \end{pmatrix}
\begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix}
= \begin{pmatrix} 0 & 2 \end{pmatrix}.
$$

So $(2,0)\to(0,2)$: a point on the $+x$ axis swings up to the $+y$ axis. ✅

### 5. Worked example — the notebook's $45^\circ$ case
At $\theta = 45^\circ$: $\cos\theta = \sin\theta = \tfrac{\sqrt2}{2}\approx 0.7071$. For the square corner $(2.5,\,2.5)$:

$$
x' = 2.5(0.7071) - 2.5(0.7071) = 0, \qquad
y' = 2.5(0.7071) + 2.5(0.7071) \approx 3.536.
$$

So $(2.5,\,2.5)\to(0,\,3.536)$. It does **not** spin in place — rotation is **about the origin**, so the square swings to a new location. Its distance from the origin, $\sqrt{2.5^2+2.5^2}\approx 3.536$, is **preserved** (a good sanity check, since rotations are rigid).

### 6. Properties to remember
- **Rigid / isometry:** preserves all distances and angles; shape and size unchanged.
- **About the origin only.** To rotate about a point $\mathbf{c}$: translate by $-\mathbf{c}$, rotate, translate back by $+\mathbf{c}$.
- **Orthogonal:** $R^\top = R^{-1}$ and $\det R = +1$. The inverse rotation is $R(-\theta)$.
- **Linear:** unlike translation, rotation fixes the origin ($R\mathbf{0}=\mathbf{0}$).

### 7. The code (for reference)

In [ ]:
import numpy as np

def rotate_polygon(polygon, theta):
    rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
                                [np.sin(theta),  np.cos(theta)]])
    return polygon @ rotation_matrix.T   # row-vector layout -> right-multiply by R^T

# sanity check: (2,0) rotated 90 deg -> (0,2)
print(rotate_polygon(np.array([[2.0, 0.0]]), np.pi/2))

---
## Ergänzung — Was bedeutet `@` und warum `.T`? (Task 1.2)

Kurze Vertiefung zu der Zeile `return polygon @ rotation_matrix.T`, in zwei Teilen: erst der Operator `@`, dann das `.T`.

### A. Der Operator `@` — Matrixmultiplikation

In NumPy/Python ist `@` der **Matrixmultiplikations-Operator** (Zeile-mal-Spalte, lineare Algebra). Wichtig ist der Unterschied:

| Operator | Bedeutung | Beispiel |
|---|---|---|
| `A @ B` | **Matrixmultiplikation** (lineare Algebra) | das wollen wir hier |
| `A * B` | **elementweise** Multiplikation (jedes Element einzeln) | **falsch** für Rotation |

Beispiel für den Unterschied:

$$
\begin{pmatrix} a & b \\ c & d \end{pmatrix} @ \begin{pmatrix} e & f \\ g & h \end{pmatrix}
= \begin{pmatrix} ae+bg & af+bh \\ ce+dg & cf+dh \end{pmatrix}
\quad\text{vs.}\quad
\begin{pmatrix} a & b \\ c & d \end{pmatrix} * \begin{pmatrix} e & f \\ g & h \end{pmatrix}
= \begin{pmatrix} ae & bf \\ cg & dh \end{pmatrix}.
$$

Nur `@` mischt die Komponenten so, wie es eine Rotation verlangt — `*` würde jeden Wert nur isoliert skalieren.

### B. Das `.T` — der Dimensions-Check

`.T` transponiert eine Matrix (Zeilen ↔ Spalten). Warum brauchen wir es?

Die Mathe-Formel $\mathbf{p}' = R\,\mathbf{p}$ erwartet den Punkt als **Spalte**. Unser `polygon` speichert die Punkte aber als **Zeilen**, Form `(N, 2)`:

```
polygon = [[x1, y1],      # Punkt 1  (eine Zeile)
           [x2, y2],      # Punkt 2
           ...      ]     # shape (N, 2)
```

Mit der Regel $(R\mathbf{p})^\top = \mathbf{p}^\top R^\top$ dreht man die Formel auf das Zeilen-Layout um: $P' = P\,R^\top$. Genau das ist `polygon @ rotation_matrix.T`.

**Shape-Check** (die innere Dimension muss übereinstimmen, hier die beiden `2`):

$$
\underbrace{(N,2)}_{\texttt{polygon}} \;@\; \underbrace{(2,2)}_{\texttt{R.T}} \;=\; \underbrace{(N,2)}_{\text{Ergebnis}}
$$

→ alle $N$ Punkte werden auf einmal rotiert, das Ergebnis hat dieselbe Form wie der Input.

> ⚠️ **Falle:** `polygon @ R` (ohne `.T`) läuft fehlerfrei durch — die Dimensionen passen ja auch — rechnet aber mit $R$ statt $R^\top$ und dreht damit **im Uhrzeigersinn** ($-\theta$ statt $+\theta$). Das Bild sieht „gedreht“ aus, nur in die falsche Richtung.

In [ ]:
import numpy as np

# @ vs * — derselbe Input, anderes Ergebnis
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print("A @ B (Matrixmult.):\n", A @ B)
print("A * B (elementweise):\n", A * B)

# .T richtig vs. falsch herum
theta = np.pi / 2  # 90 Grad
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
p = np.array([[2.0, 0.0]])         # ein Punkt als Zeile, shape (1, 2)
print("richtig  p @ R.T ->", p @ R.T)   # (2,0) -> (0,2)  CCW ✓
print("falsch   p @ R   ->", p @ R)     # (2,0) -> (0,-2) CW  ✗
print("shapes:", p.shape, "@", R.T.shape, "->", (p @ R.T).shape)

---
## Task 1.3 — Homogeneous Transformation Matrix (detailed)

### 1. The problem it solves
Translation is an **addition** ($\mathbf{p}+\mathbf{t}$); rotation is a **multiplication** ($R\mathbf{p}$). Mixing "multiply then add then multiply…" along a robot is clumsy and error-prone. **Homogeneous coordinates** let us write *both* rotation and translation as a **single matrix multiply**, so any sequence of moves composes by just multiplying matrices.

The trick: append a $1$ to every point, $(x,y)\to(x,y,1)$, and use a $3\times 3$ matrix.

### 2. The matrix
$$
T = \begin{pmatrix} \cos\theta & -\sin\theta & t_x \\ \sin\theta & \cos\theta & t_y \\ 0 & 0 & 1 \end{pmatrix}
= \begin{pmatrix} R & \mathbf{t} \\ \mathbf{0}^\top & 1 \end{pmatrix}.
$$

It has a clear block structure:
- top-left $2\times2$ block $R$ → the **rotation**,
- top-right column $\mathbf{t}=(t_x,t_y)^\top$ → the **translation**,
- bottom row $(0,0,1)$ → just carries the homogeneous $1$ through.

Apply it to a point $(x,y,1)^\top$:

$$
T\begin{pmatrix} x\\y\\1 \end{pmatrix}
= \begin{pmatrix} x\cos\theta - y\sin\theta + t_x \\ x\sin\theta + y\cos\theta + t_y \\ 1 \end{pmatrix}.
$$

Read it off: the body is **rotated first**, then the translation $\mathbf{t}$ is **added**. The third coordinate stays $1$, so we can strip it off afterward and recover a normal $(x,y)$ point.

### 3. The two functions in code

**`create_transform(theta, dx, dy)`** — just builds the $3\times3$ matrix above:
```python
np.array([[cos θ, -sin θ, dx],
          [sin θ,  cos θ, dy],
          [0,      0,      1]])
```

**`apply_transform(polygon, T)`** — applies it to all $N$ vertices in three steps:

1. **Pad to homogeneous coords:** turn the $(N,2)$ array into $(N,3)$ by appending a column of ones (`np.hstack([polygon, np.ones((N,1))])`). Each row becomes $(x,y,1)$.
2. **Multiply:** `polygon_h @ T.T`. Same row-vector reason as in Task 1.2 — our points are **rows**, so we right-multiply by $T^\top$ instead of computing $T\mathbf{p}$. Result is $(N,3)$.
3. **Drop the 1:** slice `[:, :2]` to return a clean $(N,2)$ array.

$$
\underbrace{(N,3)}_{\text{padded}} \;\;@\;\; \underbrace{(3,3)}_{T^\top} \;=\; \underbrace{(N,3)}_{\text{transformed}} \;\xrightarrow{[:, :2]}\; \underbrace{(N,2)}_{\text{result}}.
$$

### 4. Worked example — the notebook's case ($\theta=45^\circ$, $\mathbf{t}=(2,2)$)

Build $T = $ `create_transform(π/4, 2, 2)` with $\cos45^\circ=\sin45^\circ\approx0.7071$:

$$
T = \begin{pmatrix} 0.7071 & -0.7071 & 2 \\ 0.7071 & 0.7071 & 2 \\ 0 & 0 & 1 \end{pmatrix}.
$$

Transform the square corner $(2.5,\,2.5)$, i.e. homogeneous $(2.5,2.5,1)$:

$$
\begin{aligned}
x' &= 2.5(0.7071) - 2.5(0.7071) + 2 = 0 + 2 = 2,\\
y' &= 2.5(0.7071) + 2.5(0.7071) + 2 \approx 3.536 + 2 = 5.536.
\end{aligned}
$$

So $(2.5,2.5)\to(2,\,5.536)$. Compare with the **pure rotation** result from Task 1.2 which gave $(0,\,3.536)$: it's exactly that, **shifted by $\mathbf{t}=(2,2)$**. This confirms the order: **rotate first, then translate.**

---
## Ergänzung — Task 1.3: Warum die `1`, und `@ T.T`?

Vertiefung zu `apply_transform`. Zwei Fragen: warum hängen wir an jeden Punkt eine **`1`** an, und warum wieder **`@ T.T`**?

### A. Warum die zusätzliche `1` (homogene Koordinaten)?

Rotation ist eine **Multiplikation** ($R\mathbf{p}$), Translation eine **Addition** ($\mathbf{p}+\mathbf{t}$). Eine reine $2\times2$-Matrix kann nicht addieren — sie bildet den Ursprung immer auf sich selbst ab. Mit dem Anhängen einer `1` wird die Translation zu einem Teil **einer einzigen Matrixmultiplikation**:

$$
\begin{pmatrix} \cos\theta & -\sin\theta & t_x \\ \sin\theta & \cos\theta & t_y \\ 0 & 0 & 1 \end{pmatrix}
\begin{pmatrix} x \\ y \\ 1 \end{pmatrix}
= \begin{pmatrix} x\cos\theta - y\sin\theta + t_x \\ x\sin\theta + y\cos\theta + t_y \\ 1 \end{pmatrix}.
$$

Die `1` in der letzten Zeile „aktiviert“ die Translationsspalte $t_x, t_y$. Genau das macht `np.hstack([polygon, np.ones((N,1))])`: aus jedem $(x,y)$ wird $(x,y,1)$.

Am Ende schneidet `[:, :2]` die `1` wieder ab → zurück zu normalen $(x,y)$-Punkten.

### B. Warum `@ T.T` (gleiche Logik wie 1.2) und der Shape-Fluss

Auch hier liegen die Punkte als **Zeilen** vor, also rechnen wir $P' = P\,T^\top$ statt $T\mathbf{p}$ — daher wieder `.T`. Der einzige Unterschied zu 1.2: die Matrizen sind jetzt $3\times3$ und die Punkte $(N,3)$.

$$
\underbrace{(N,3)}_{\text{padded}} \;@\; \underbrace{(3,3)}_{T^\top} \;=\; \underbrace{(N,3)}_{\text{transformed}} \;\xrightarrow{[:,\,:2]}\; \underbrace{(N,2)}_{\text{Ergebnis}}.
$$

**Lese-Reihenfolge der Operation:** erst rotieren, dann translieren — die Translation steckt in der dritten Spalte und wird *nach* der Rotation addiert. Das ist die Brücke zu Task 2: weil alles **eine** Matrix ist, lassen sich mehrere solcher Schritte einfach **multiplizieren**.

In [ ]:
import numpy as np

def create_transform(theta, dx, dy):
    return np.array([[np.cos(theta), -np.sin(theta), dx],
                     [np.sin(theta),  np.cos(theta), dy],
                     [0, 0, 1]])

def apply_transform(polygon, T):
    N = polygon.shape[0]
    h = np.hstack([polygon, np.ones((N, 1))])   # (N,2) -> (N,3): jede Zeile (x,y,1)
    return (h @ T.T)[:, :2]                      # multiplizieren, dann die 1 abschneiden

# Check: Punkt (1,0) um 90 Grad drehen und um (2,2) verschieben -> erst Rotation (1,0)->(0,1), dann +（2,2) = (2,3)
T = create_transform(np.pi/2, 2, 2)
print(apply_transform(np.array([[1.0, 0.0]]), T))   # ~ [2. 3.]

---
# Task 2 — Kinematic Chains in 2D (detailed)

### 1. Die Idee: ein Frame pro Link
Ein Roboterarm besteht aus mehreren **Links** (starre Glieder), verbunden durch **Gelenke**. Jeder Link hat sein **eigenes Körper-Koordinatensystem (Frame)**. Ein Punkt wird im Frame seines Links beschrieben — die Aufgabe ist, ihn in **Weltkoordinaten** umzurechnen.

Das Werkzeug dafür ist genau die homogene Transformation aus Task 1.3: jeder Link-Frame ist über eine $3\times3$-Matrix relativ zu seinem **Eltern-Link** definiert.

### 2. Die Kette: Transformationen multiplizieren — Reihenfolge zählt!
Um vom Frame des letzten Links bis in die Welt zu kommen, **multipliziert man die Transformationen entlang der Kette**:

$$
\mathbf{p}_{\text{welt}} \;=\; T_1\,T_2\,\cdots\,T_k \; \mathbf{p}_{\text{körper}}.
$$

Mit Spaltenvektoren wirkt das **von rechts nach links**: ganz rechts steht der Punkt, dann wird $T_k$ angewandt (Link relativ zu seinem Elter), … zuletzt $T_1$ (Basis relativ zur Welt).

**Faustregel für Eltern–Kind:** der **Eltern-Link steht links**. Wenn $T_{A_1}$ die Welt-Pose von Link $A_1$ ist und $T_{\text{rel}}$ beschreibt, wie $A_2$ relativ zu $A_1$ sitzt, dann

$$
T_{A_2} \;=\; T_{A_1}\; T_{\text{rel}}.
$$

> ⚠️ **$AB \neq BA$.** Matrixmultiplikation ist nicht kommutativ — „erst drehen, dann schieben“ ergibt etwas anderes als „erst schieben, dann drehen“.

### 3. Der Zwei-Link-Arm Schritt für Schritt
Gegeben Basislänge $l_1$, Armlänge $l_2$, Gelenkwinkel $\theta$:

1. **$A_1$** als Rechteck um den Ursprung bauen und mit $T_{A_1}$ in die Welt setzen (z. B. Basis bei $(3,5)$).
2. **Kette für $A_2$** (Eltern $A_1$ **links**, dann von links nach rechts die relativen Schritte):
$$
T_{A_2} = T_{A_1}\;\cdot\;\underbrace{\mathrm{Trans}(l_1/2,\,0)}_{\text{zum Gelenk am Ende von }A_1}\;\cdot\;\underbrace{\mathrm{Rot}(\theta)}_{\text{Gelenk}}\;\cdot\;\underbrace{\mathrm{Trans}(l_2/2,\,0)}_{\text{zur Mitte von }A_2}.
$$
3. $T_{A_2}$ auf das $A_2$-Rechteck anwenden und beide plotten.

Weil $A_1$ **um den Ursprung zentriert** ist, liegt sein Endgelenk bei $+l_1/2$ (nicht $l_1$). Und der letzte Schritt $\mathrm{Trans}(l_2/2,0)$ ist nötig, damit $A_2$ vom Gelenk **wegragt**, statt mittig auf dem Gelenk zu sitzen.

### 4. ⚠️ Häufige Falle — Reihenfolge im aktuellen Code
Die Lösung im Aufgaben-Notebook verwendet

```python
T_A2 = create_transform(joint_angle, dx=base_length, dy=0) @ T_A1   # T_A1 steht RECHTS
```

Hier steht der Eltern-Frame $T_{A_1}$ **rechts** statt links. Folge: $A_2$ wird **um den Welt-Ursprung** gedreht statt um das Gelenk. Numerisch (mit $l_1=4,\ \theta=45^\circ$, Basis bei $(3,5)$):

| Reihenfolge | Mitte von $A_2$ |
|---|---|
| `create_transform(θ,4,0) @ T_A1` (Code) | $(2.59,\ 5.66)$ — **links** von $A_1$ ❌ |
| `T_A1 @ create_transform(θ,4,0)` (Eltern links) | $(7,\ 5)$ — rechts, am Ende von $A_1$ ✓ |

$A_2$ sollte am **rechten Ende** von $A_1$ ansetzen, nicht links davon. Zusätzlich gilt: mit zentrierter Basis sollte zum Gelenk um $l_1/2$ (=2) verschoben werden, und ein abschließendes $\mathrm{Trans}(l_2/2,0)$ fehlt, damit der Arm vom Gelenk wegragt.

### 5. Korrigierte Kette (zum Vergleich)

In [ ]:
import numpy as np

def draw_two_link_arm_fixed(base_length, arm_length, joint_angle):
    height = 0.5
    rect = lambda L: np.array([(-L/2,-height/2),(L/2,-height/2),(L/2,height/2),(-L/2,height/2)])

    T_A1 = create_transform(0.0, 3, 5)                 # Basis-Pose in der Welt
    A1_world = apply_transform(rect(base_length), T_A1)

    # Eltern LINKS, dann: zum Gelenk (l1/2) -> drehen -> zur Mitte von A2 (l2/2)
    T_A2 = (T_A1
            @ create_transform(0.0, base_length/2, 0)   # ans Ende von A1
            @ create_transform(joint_angle, 0, 0)       # Gelenk drehen
            @ create_transform(0.0, arm_length/2, 0))   # zur Mitte von A2
    A2_world = apply_transform(rect(arm_length), T_A2)
    return A1_world, A2_world

_, A2 = draw_two_link_arm_fixed(4.0, 3.0, np.pi/4)
print('A2 Eckpunkte (Welt):')
print(np.round(A2, 3))   # sitzt rechts von A1 und ragt unter dem Winkel weg

---
### 6. Die Zeile `T_A2 = create_transform(joint_angle, dx=base_length, dy=0) @ T_A1` — zerlegt

**Die Bausteine:**

- `create_transform(joint_angle, dx=base_length, dy=0)` baut **eine** $3\times3$-Matrix $M$, die zwei Dinge zugleich kodiert: um `joint_angle` **drehen** und um $(l_1, 0)$ entlang der x-Achse **verschieben** ("zum Gelenk am Ende von $A_1$ laufen"):

$$
M = \begin{pmatrix} \cos\theta & -\sin\theta & l_1 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{pmatrix}
$$

- `T_A1` ist die **Welt-Pose von $A_1$** (hier: Basis bei $(3,5)$, keine Drehung).
- `@` ist Matrixmultiplikation, also `T_A2 = M @ T_A1`.

**Was `M @ T_A1` mit einem Punkt macht.** Bei `apply_transform(polygon_A2, T_A2)` wird jeder $A_2$-Eckpunkt $\mathbf{p}$ (im Körper-Frame von $A_2$) zu

$$
\mathbf{p}_{\text{welt}} = T_{A_2}\,\mathbf{p} = M\,(T_{A_1}\,\mathbf{p}).
$$

Von **rechts nach links** gelesen:
1. **zuerst** `T_A1` → schiebt $\mathbf{p}$ in die Gegend um $(3,5)$;
2. **dann** `M` → dreht diesen bereits verschobenen Punkt **um den Welt-Ursprung** $(0,0)$ und schiebt ihn um $(l_1,0)$.

**Genau das ist das Problem:** weil `T_A1` **rechts** steht, dreht die Rotation um den Welt-Ursprung statt um das **Gelenk**. $A_2$ schwingt damit um $(0,0)$ herum, statt am Ende von $A_1$ anzusetzen.

**Regel:** Eltern-Frame gehört **nach links** — `T_A2 = T_A1 @ M`. Dann heißt es: erst $A_1$-Pose, *dann* lokal zum Gelenk und drehen.

| Ausdruck | Mitte von $A_2$ ($l_1{=}4,\ \theta{=}45^\circ$, Basis $(3,5)$) | |
|---|---|---|
| `M @ T_A1` (aktueller Code) | $(2.59,\ 5.66)$ | links von $A_1$ ❌ |
| `T_A1 @ M` (Eltern links) | $(7,\ 5)$ | rechtes Ende von $A_1$ ✓ |

(Korrigierte vollständige Kette siehe Abschnitt 5 oben.)

---
### 7. Einfach erklärt (für Einsteiger) — dein eigener Arm 🦾

Stell dir deinen **eigenen Arm** vor: **Oberarm** = Link $A_1$, **Unterarm** = Link $A_2$, **Ellbogen** = das Gelenk.

Um den Unterarm an die richtige Stelle zu zeichnen, machst du **der Reihe nach**:

1. **Bei der Schulter starten** → da sitzt $A_1$ (im Code: Position $(3,5)$).
2. **Am Oberarm entlang bis zum Ellbogen laufen** → ans Ende von $A_1$.
3. **Ellbogen beugen** → um den Gelenkwinkel drehen.
4. **Unterarm vom Ellbogen weg ausstrecken** → entlang der Länge von $A_2$.

Jeder Schritt beginnt **dort, wo der vorige aufgehört hat**.

**Was der ursprüngliche Code stattdessen macht:**

```python
T_A2 = create_transform(joint_angle, dx=base_length, dy=0) @ T_A1
```

Bei `@` liest der Computer **von rechts nach links**. Es passiert also:
1. $A_2$ wird zur Schulter $(3,5)$ gesetzt (`T_A1` ist rechts → läuft zuerst).
2. Dann wird alles um die **Ecke der Landkarte $(0,0)$** gedreht — *nicht* um den Ellbogen.

Das ist, als würdest du sagen: *„Stell dich an die Schulter und dreh dann den ganzen Unterarm um einen Punkt 5 Meter weiter am Boden."* Der Unterarm fliegt an die falsche Stelle. 👎

**Die Reparatur — `T_A1` einfach nach LINKS (zuerst):**

```python
T_A2 = (T_A1                                       # 1. an der Schulter starten
        @ create_transform(0.0, base_length/2, 0)  # 2. zum Ellbogen laufen
        @ create_transform(joint_angle, 0, 0)      # 3. Ellbogen beugen
        @ create_transform(0.0, arm_length/2, 0))   # 4. Unterarm ausstrecken
```

Jetzt liest man es **links → rechts**, genau wie die 4 Schritte oben.

**Die eine Regel zum Merken:**
> **Der Eltern-Link ($A_1$) steht LINKS.** Baue die Kette in der Reihenfolge, in der du die Bewegungen wirklich machst: Schulter → Ellbogen → beugen → Unterarm.

**Zwei kleine Extras, die im Code fehlten:**
- $A_1$ ist **um den Ursprung zentriert**, also liegt der Ellbogen bei `base_length/2`, nicht `base_length` → nur die **halbe** Länge laufen.
- Schritt 4 (`arm_length/2`) fehlte → der Unterarm saß *auf* dem Ellbogen statt davon *wegzuragen*.

---
# Task 3.1 — GridMap (einfach erklärt)

✅ **Deine Lösung ist korrekt und vollständig** — sie läuft und verhält sich genau wie der Test erwartet. Hier trotzdem, was jeder Teil tut.

### 1. Was ist ein Occupancy Grid?
Stell dir **kariertes Papier** über der Welt vor. Jedes Kästchen (**Zelle**) ist entweder:
- **frei** → `0` (Roboter darf hin)
- **belegt** → `1` (da steht ein Hindernis)

Damit wird aus „Wo darf der Roboter hin?" ein einfaches Raster aus Nullen und Einsen — perfekt für die Suche in Task 4.

### 2. Der Konstruktor — das Papier anlegen
```python
self.grid = np.zeros((height, width), dtype=np.uint8)
```
- Start: überall `0` (alles frei).
- `resolution` = **wie viele Meter eine Zelle ist** (z. B. `0.1` = 10 cm pro Kästchen). Kleiner = feiner, aber langsamer.
- `uint8` = winzige Ganzzahlen (0–255), spart Speicher, da wir nur 0/1 brauchen.

### 3. ⚠️ Der verwirrende Teil: `[y, x]` statt `[x, y]`
Ein NumPy-Array wird mit **`[Zeile, Spalte]`** indiziert. Im Bild:
- **Zeilen** gehen hoch/runter → das ist die **y**-Richtung
- **Spalten** gehen seitwärts → das ist die **x**-Richtung

Die Zelle an Position `(x, y)` liegt deshalb bei **`self.grid[y, x]`** — die Reihenfolge ist **vertauscht**. Genau darum ist die Form `(height, width)` und nicht `(width, height)`.

```python
def set_obstacle(self, x, y):
    if 0 <= x < self.width and 0 <= y < self.height:   # im Papier bleiben
        self.grid[y, x] = 1                            # Achtung: [y, x]
```

### 4. `is_free` — zwei Sicherheits-Checks
```python
def is_free(self, x, y):
    if not (0 <= x < self.width and 0 <= y < self.height):
        return False              # ausserhalb der Karte = nicht frei
    return self.grid[y, x] == 0   # frei nur wenn die Zelle 0 ist
```
Zwei Dinge machen eine Zelle *nicht* frei: sie liegt **ausserhalb der Karte** oder sie ist **belegt**. „Ausserhalb = blockiert" ist clever — so plant der Roboter keine Wege, die die Welt verlassen.

### 5. `visualize` — das Zeichnen
- `imshow(..., origin="lower")` setzt `(0,0)` **unten links**, wie in einem normalen Mathe-Diagramm (ohne das Argument steht das Bild kopfüber).
- `path` und `robot_pos` zeichnen nur Linien/Punkte oben drauf, falls man sie übergibt (gebraucht in Task 4).
- die `minor`-Ticks malen die feinen Gitterlinien, damit man einzelne Zellen sieht.

### 6. Warum der Test besteht
```python
for y in range(10): gm.set_obstacle(5, y)   # x bleibt 5, y läuft -> senkrechte Wand
gm.is_free(4, 4)  # True  — Spalte 4 ist leer
gm.is_free(5, 4)  # False — Spalte 5 ist die Wand
```
Eine senkrechte Wand heisst **x bleibt 5**, während **y variiert** — genau das macht die Schleife. 👍

---
# Task 3.2 — add_polygon_obstacle_to_grid (einfach erklärt)

✅ **Deine Lösung ist korrekt** — das 0.5 m × 0.5 m Quadrat füllt bei 0.1 m Auflösung exakt 25 Zellen, genau wie erwartet.

### 1. Das Ziel: eine Form in Kästchen umwandeln
Ein Hindernis ist als **Polygon** beschrieben (echte Form mit glatten Kanten). Dein Grid kennt aber nur **Kästchen (Zellen)**. „Rastern" heisst: *welche Kästchen male ich aus, um die Form darzustellen?*

Die gewählte Regel: **male eine Zelle aus, wenn das Polygon ihren Mittelpunkt überdeckt.** 🎯

### 2. Wie der Code das macht
```python
for iy in range(gridmap.height):
    for ix in range(gridmap.width):
        cx = (ix + 0.5) * gridmap.resolution
        cy = (iy + 0.5) * gridmap.resolution
        if polygon.contains(Point(cx, cy)):
            gridmap.set_obstacle(ix, iy)
```

1. **Jede Zelle besuchen** — die Doppelschleife läuft über alle `(ix, iy)`.
2. **Den Mittelpunkt der Zelle in Metern finden:**
   - `ix * resolution` = **linke Kante** der Zelle in Metern,
   - `+ 0.5` = in die **Mitte** der Zelle rücken,
   - `* resolution` rechnet von „Zellennummer" in „Meter" um.
   - Beispiel: `ix=22` bei `resolution=0.1` → `(22+0.5)*0.1 = 2.25 m`.
3. **shapely fragen:** `polygon.contains(Point(cx, cy))` → `True`, wenn der Punkt im Polygon liegt (shapely macht die Geometrie-Mathematik).
4. **Wenn ja → als Hindernis markieren** mit `set_obstacle(ix, iy)`.

### 3. Die zentrale Idee: Gitter- ↔ Welt-Koordinaten
Das ist die Brücke zwischen zwei „Sprachen":
- **Gitter-Sprache:** Zellindizes `(ix, iy)` — ganze Zahlen wie Zelle 22
- **Welt-Sprache:** echte Positionen in Metern — wie 2.25 m

`(ix + 0.5) * resolution` ist der **Übersetzer** von Gitter → Welt. Das Polygon lebt in **Metern**, das Gitter in **Zellindizes** — also muss man *vor* der shapely-Frage umrechnen.

**Warum „+0.5"?** Mit nur `ix * resolution` würde man die **Ecke** der Zelle testen, nicht die **Mitte**. Die Mitte ist fairer: eine Zelle gilt nur als blockiert, wenn das Hindernis ihre Mitte wirklich überdeckt → sauberere, symmetrische Rasterung.

### 4. Trade-off (gut für Prüfungsfragen)
Diese Rasterung ist **verlustbehaftet (lossy)**:
- Ein dünnes Hindernis, das *zwischen* den Zellmittelpunkten durchrutscht, markiert evtl. **keine** Zelle.
- Das blockige Ergebnis ist nur eine **Näherung** der glatten Form.

Feinere `resolution` (kleinere Zellen) = genauer, aber mehr Zellen in der Schleife = **langsamer**. Das ist der klassische Auflösungs-Kompromiss aus Task 3.